# 04 · Linear regression as a sanity baseline

A single global `LinearRegression` over all series via `mlforecast` — same 
lag/calendar features the LightGBM model uses, just a linear head. Cheap 
and reveals whether a tree model is buying anything beyond linear structure.

Note: the original notebook fit one regression per SKU. That's slow and 
throws away cross-series signal. The global model here matches the package's 
design tenet.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import pandas as pd
from mlforecast import MLForecast
from mlforecast.lag_transforms import RollingMean
from mlforecast.target_transforms import Differences
from sklearn.linear_model import LinearRegression

from m5.config import SETTINGS, set_global_seed
from m5.evaluation import compute_components, wrmsse_for_models
from m5.features import build_feature_frame, static_features
from m5.models.lgbm import DEFAULT_LAGS, DEFAULT_ROLLS
from m5.plots import configure_style

configure_style()
set_global_seed()

## Load + build features

In [ ]:
df = build_feature_frame(pd.read_parquet(SETTINGS.processed_dir / 'long.parquet'))
statics = static_features(df)
static_cols = [c for c in ('item_id', 'dept_id', 'cat_id', 'store_id', 'state_id') if c in statics.columns]
feature_cols = [c for c in ('snap', 'is_event', 'price_norm', 'price_change_pct') if c in df.columns]
keep_cols = ['unique_id', 'ds', 'y', *feature_cols]
df_fit = df[keep_cols].copy()
df_fit.head()

## Build the global linear forecaster

Same lag/rolling skeleton as `m5.models.lgbm.build_lgbm_forecaster` so the 
scores are directly comparable — only the head changes.

In [ ]:
fcst = MLForecast(
    models={'Linear': LinearRegression()},
    freq='D',
    lags=list(DEFAULT_LAGS),
    lag_transforms={1: [RollingMean(window_size=w) for w in DEFAULT_ROLLS]},
    date_features=['dayofweek', 'day', 'week', 'month', 'year'],
    target_transforms=[Differences([1])],
)

## Cross-validation + score

In [ ]:
cv = fcst.cross_validation(
    df=df_fit,
    h=SETTINGS.horizon,
    n_windows=SETTINGS.n_windows,
    static_features=static_cols,
)
cv.to_parquet(SETTINGS.artifacts_dir / 'cv_linear.parquet', index=False)
cv.tail()

In [ ]:
components = compute_components(df[df['ds'] < cv['ds'].min()])
wrmsse_for_models(cv[['unique_id', 'ds', 'y']], cv, components)